# Virtual stores with the `vds` plugin

This notebook replaces the old Icechunk tutorial: build a virtual store from NASA
granules, read it back, and append new granules — **without copying any data**.

Instead of the low-level `virtualize()` / `write_virtual()` / `open_virtual()`
calls, everything goes through one request:

```python
job = ea.plugins.vds.request(granules, store="mur_sst.icechunk", concat_dim="time")
asset = job.results()[0]      # the store, as an Asset
ds = ea.open(asset.href)      # read it back
```

The request returns a `Job` immediately, so you can watch it work
(`job.info()`), open the store when it is done, and call the same request again
with a new batch to **append** — the store's existence decides create vs append.

Install the plugin first (it is not on PyPI yet):

```bash
pip install "earthaccess-plugin-vds[distributed] @ git+https://github.com/betolink/earthaccess-plugin-vds"
```

The `distributed` extra also installs `dask-labextension`, so the Dask
cluster and its dashboard appear in the JupyterLab launcher and sidebar
(restart JupyterLab after installing).


In [1]:
import earthaccess

earthaccess.login()

Auth(authenticated=True, user='earthaccess', strategy='netrc', system=urs.earthdata.nasa.gov)

## 1. Build a virtual dataset from granules

MUR SST is a level-4, globally gridded SST product — a good fit for
virtualizing: the granules are homogeneous, CF-compliant, and share a unified
grid.

In [2]:
sst_granules = earthaccess.search_data(
    concept_id="C1996881146-POCLOUD", temporal=("2013-01-02", "2013-01-31")
)
len(sst_granules)

20

## 2. Create the store and watch the request

`request()` returns a `Job` right away. While it runs, `job.info()` reports
which phase it is in — `virtualize` (opening granules), `combine`, or `write` —
plus how many granules have been opened, the total, and the percent complete.
`show_progress=True` also draws a per-granule progress bar.

The result is an `Asset` whose `href` is the store (`format="icechunk"`,
`access="file"` for a local directory). Nothing was downloaded: the store holds
byte-range references to the original NASA files.

In [3]:
import time

TERMINAL = {"SUCCEEDED", "PARTIAL", "FAILED", "CANCELED", "PAUSED"}


def wait_with_progress(job, interval=2.0):
    """Poll a job and print its phase + per-granule counts while it runs."""
    while job.status().name not in TERMINAL:
        info = job.info()
        print(
            f"{info['phase']:>10}  {info['opened']}/{info['total']} granules"
            f"  {info['percent']:5.1f}%"
        )
        time.sleep(interval)
    info = job.info()
    print(
        f"{job.status().name}  {info['opened']}/{info['total']} granules"
        f"  {info['percent']:5.1f}%"
    )
    if info.get("error"):
        print(info["error"])
    return job

In [4]:
job = earthaccess.plugins.vds.request(
    sst_granules,
    store="mur_sst.icechunk",
    concat_dim="time",
    show_progress=True,
)
wait_with_progress(job)

virtualize  0/31 granules    0.0%
FAILED  0/31 granules    0.0%


<VirtualJob vds:3adbb5b03b964166a36d218f25e33484 FAILED>

In [ ]:
asset = job.results()[0]
asset

## 3. Read the store back

`ea.open()` recognizes an icechunk store (from the `Asset`'s `format` or the
path itself), authorizes its virtual chunk containers with your EDL
credentials, and returns an xarray dataset. Reads are served straight from the
original NASA files.

In [ ]:
ds = earthaccess.open(asset.href)
print(f"Dataset spanning from {ds.time.min().values} to {ds.time.max().values}")
ds

`ds` is a concrete, lazily-loaded dataset. Pass `load=False` to see the
manifest form instead (variables backed by `ManifestArray`):

```python
earthaccess.open(asset.href, load=False)
```


## 4. Append the next month

Search the following time step and send **the same request**, with
`store=asset.href`. Because the store already exists, the write is an append
along `concat_dim` (`append_dim` defaults to it) — no flag needed. The href
stays the same and icechunk records the update as a new commit on `main`.

In [ ]:
new_granules = earthaccess.search_data(
    concept_id="C1996881146-POCLOUD", temporal=("2013-02-01", "2013-02-28")
)
len(new_granules)

In [ ]:
append_job = earthaccess.plugins.vds.request(
    new_granules,
    store=asset.href,
    concat_dim="time",
    show_progress=True,
)
wait_with_progress(append_job)

asset = append_job.results()[0]
asset

In [ ]:
updated_ds = earthaccess.open(asset.href)
print(
    f"Dataset now spanning from {updated_ds.time.min().values} to "
    f"{updated_ds.time.max().values}"
)

## 5. The same workflow with kerchunk-parquet

`format="kerchunk"` writes a references **directory** whose name ends in
`.parquet` (that is what `ea.open()` sniffs). Kerchunk has no in-place append:
appending reads the old references, combines them with the delta, and writes a
**new versioned directory** — `mur_sst_refs.parquet` → `mur_sst_refs_v2.parquet`
— so the old version is preserved. Pass `replace=True` to rewrite in place and
keep a stable href.

In [ ]:
kerchunk_job = earthaccess.plugins.vds.request(
    sst_granules,
    store="mur_sst_refs.parquet",
    format="kerchunk",
    concat_dim="time",
    show_progress=True,
)
wait_with_progress(kerchunk_job)

kerchunk_asset = kerchunk_job.results()[0]
kerchunk_asset

In [ ]:
kerchunk_ds = earthaccess.open(kerchunk_asset)
print(
    f"Kerchunk store spanning from {kerchunk_ds.time.min().values} to "
    f"{kerchunk_ds.time.max().values}"
)

In [ ]:
kerchunk_append = earthaccess.plugins.vds.request(
    new_granules,
    store=kerchunk_asset.href,
    format="kerchunk",
    concat_dim="time",
    show_progress=True,
)
wait_with_progress(kerchunk_append)

kerchunk_asset = kerchunk_append.results()[0]
kerchunk_asset.href  # a new versioned directory, e.g. mur_sst_refs_v2.parquet

In [ ]:
kerchunk_updated = earthaccess.open(kerchunk_asset)
print(
    f"Kerchunk store now spanning from "
    f"{kerchunk_updated.time.min().values} to {kerchunk_updated.time.max().values}"
)

## Progress and cancellation

- `job.info()` always returns the same shape:
  `{"status", "opened", "total", "failed", "percent", "phase", "dashboard"}`.
- `phase` goes `virtualize → combine → write → done`.
- `show_progress=True` draws a `tqdm` bar for the per-granule opens;
  `show_progress=None` (the default) enables it in notebooks and terminals.
- When a distributed `Client` is active, `info()["dashboard"]` links to its
  dashboard. (The open step currently runs in this process so credentials never
  cross the Dask wire.)
- For cluster-side work, hand the workers their environment explicitly:
  `ea.distribute_credentials({...}, cluster=cluster)` sets `EARTHDATA_TOKEN`
  (from your login) and any AWS keys you pass. It uses Coiled's private
  `send_private_envs` channel; a plain Dask cluster needs `private=False`
  to opt into sending the values through the scheduler. The result contains
  the key names only, never the values.
- `job.cancel()` cancels per-granule opens that have not started yet.

```python
job = earthaccess.plugins.vds.request(sst_granules, store="tmp.icechunk", concat_dim="time")
job.info()      # {'phase': 'virtualize', 'opened': 3, 'total': 30, 'percent': 10.0, ...}
job.cancel()    # True if any pending granule was cancelled
```


## Summary

| Old API | `vds` plugin |
|---|---|
| `ea.virtualize(granules, concat_dim="time")` | `ea.plugins.vds.request(granules, concat_dim="time").results()` |
| `ea.write_virtual(vds, "store")` | `ea.plugins.vds.request(granules, store="store.icechunk", concat_dim="time").results()` → `[Asset]` |
| `ea.write_virtual(vds, "store", append_dim="time")` | the same request against the existing store (`append_dim` defaults to `concat_dim`) |
| `ea.open_virtual("store")` | `ea.open(asset.href)` (or `ea.open(asset.href, load=False)`) |
| `format="kerchunk"` reference file | `format="kerchunk"` with a `*.parquet` directory; appends make a versioned sibling |

The store is versioned (icechunk): every update is a new commit on `main`, so
you can inspect history, roll back, or share the store with collaborators who
have access to the underlying data.
